In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

# ==========================================
# 1. ĐỌC DỮ LIỆU & KHỞI TẠO TRACKING
# ==========================================
df = pd.read_csv('/kaggle/input/datasets/letatdinh/melb-data-house-pricing/melb_data.csv')
print(f"[*] DỮ LIỆU GỐC: Tổng cộng có {df.shape[0]} dòng và {df.shape[1]} cột.\n")

# ==========================================
# 1: TỐI ƯU DỮ LIỆU
# ==========================================
print("[--- (DATA CLEANING) ---]")

# 1.1 Xóa nhãn Null (Target)
missing_price = df['Price'].isnull().sum()
df.dropna(subset=['Price'], inplace=True)
print(f"[-] BƯỚC 1: Xóa {missing_price} dòng không có Giá bán (Target = Null). Còn lại: {df.shape[0]} dòng.")

# 1.2 Lọc dị biệt (Outliers)
# Theo dõi Giá (Loại bỏ giá ảo)
outlier_price = df[df['Price'] < 100000].shape[0]
df = df[df['Price'] >= 100000]
print(f"[-] BƯỚC 2: Xóa {outlier_price} giao dịch ảo (Giá < 100.000 AUD). Còn lại: {df.shape[0]} dòng.")

# Theo dõi Diện tích đất (Loại bỏ trang trại)
outlier_land = df[df['Landsize'] >= 10000].shape[0]
df = df[df['Landsize'] < 10000]
print(f"[-] BƯỚC 3: Xóa {outlier_land} mảnh đất phi lý (Rộng >= 10.000 m2). Còn lại: {df.shape[0]} dòng.")

# Theo dõi Diện tích xây dựng
# BUG FIX: Nếu chỉ dùng df['BuildingArea'] > 10, pandas sẽ xóa luôn cả những dòng bị Null (vì Null > 10 là False).
# Điều này làm mất đi hàng ngàn dữ liệu quý giá. Ta phải dùng logic: (Lớn hơn 10) HOẶC (Là Null).
outlier_building = df[df['BuildingArea'] <= 10].shape[0]
df = df[(df['BuildingArea'] > 10) | (df['BuildingArea'].isnull())]
print(f"[-] BƯỚC 4: Xóa {outlier_building} căn nhà lỗi (Diện tích xây <= 10 m2). Còn lại: {df.shape[0]} dòng.")

# 1.3 Kiểm tra tình trạng "Lỗ hổng" trước khi giao cho Pipeline
print("\n[--- TRẠNG THÁI LỖ HỔNG (MISSING DATA) TRƯỚC KHI VÀO PIPELINE ---]")
missing_building = df['BuildingArea'].isnull().sum()
missing_year = df['YearBuilt'].isnull().sum()
print(f"[*] Cột 'BuildingArea' trống {missing_building} dòng -> Middleware sẽ tự động điền Trung vị (Median).")
print(f"[*] Cột 'YearBuilt' trống {missing_year} dòng -> Middleware sẽ tự động điền Trung vị (Median).")

# 1.4 Feature Engineering
df['HouseAge'] = 2026 - df['YearBuilt']
print(f"\n[+] BƯỚC 5: Tạo thành công cột 'HouseAge' (Tuổi thọ nhà).")
print("[--- KẾT THÚC TIỀN XỬ LÝ DỮ LIỆU THÔ ---]\n")

# Lựa chọn Features
numeric_features = ['Rooms', 'Distance', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'HouseAge']
categorical_features = ['Type', 'Method', 'Regionname']

X = df[numeric_features + categorical_features]
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# MIDDLEWARE TIỀN XỬ LÝ
# ==========================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# ==========================================
# 2: TỐI ƯU THUẬT TOÁN (XGBOOST TUNING)
# ==========================================
print("Đang huấn luyện mô hình XGBoost (Engine: Gradient Boosting)...")

model_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    
    # Cấu hình "Sweet Spot" của XGBoost
    ('regressor', xgb.XGBRegressor(
        n_estimators=350,       # Số cây
        learning_rate=0.4,     # Tốc độ học
        max_depth=3,            # Độ sâu
        random_state=42,
        n_jobs=-1               # Dùng full CPU Kaggle
    ))
])

# Kích hoạt học
model_xgb.fit(X_train, y_train)

# ==========================================
# ĐÁNH GIÁ CHẤT LƯỢNG (TRAIN & TEST)
# ==========================================
# Lấy điểm Train để kiểm tra xem có bị Overfitting không
y_pred_train = model_xgb.predict(X_train)
r2_train = r2_score(y_train, y_pred_train)

# Lấy điểm Test (Thực tế)
y_pred_test = model_xgb.predict(X_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("\n" + "="*45)
print("BÁO CÁO TỐI ƯU HÓA XGBOOST")
print("="*45)
print(f"1. Độ chính xác Train (Train R2) : {r2_train * 100:.2f}%")
print(f"2. Độ chính xác Test (Test R2)  : {r2_test * 100:.2f}%")
print(f"3. Sai lệch trung bình (MAE)       : {mae_test:,.0f} AUD")
print("="*45)
if r2_train - r2_test > 0.15:
    print("Mô hình đang có dấu hiệu Overfitting. Cần giảm max_depth xuống!")
else:
    print("Hệ thống ổn định (Good Fit).")

[*] DỮ LIỆU GỐC: Tổng cộng có 13580 dòng và 21 cột.

[--- BẮT ĐẦU QUÁ TRÌNH KHỬ TRÙNG (DATA CLEANING) ---]
[-] BƯỚC 1: Xóa 0 dòng không có Giá bán (Target = Null). Còn lại: 13580 dòng.
[-] BƯỚC 2: Xóa 1 giao dịch ảo (Giá < 100.000 AUD). Còn lại: 13579 dòng.
[-] BƯỚC 3: Xóa 26 mảnh đất phi lý (Rộng >= 10.000 m2). Còn lại: 13553 dòng.
[-] BƯỚC 4: Xóa 74 căn nhà lỗi (Diện tích xây <= 10 m2). Còn lại: 13479 dòng.

[--- TRẠNG THÁI LỖ HỔNG (MISSING DATA) TRƯỚC KHI VÀO PIPELINE ---]
[*] Cột 'BuildingArea' trống 6433 dòng -> Middleware sẽ tự động điền Trung vị (Median).
[*] Cột 'YearBuilt' trống 5343 dòng -> Middleware sẽ tự động điền Trung vị (Median).

[+] BƯỚC 5: Chế tạo thành công cột 'HouseAge' (Tuổi thọ nhà).
[--- KẾT THÚC TIỀN XỬ LÝ DỮ LIỆU THÔ ---]

Đang huấn luyện mô hình XGBoost (Engine: Gradient Boosting)...

🚀 BÁO CÁO TỐI ƯU HÓA XGBOOST
1. Độ chính xác lúc Học (Train R2) : 88.55%
2. Độ chính xác Thực tế (Test R2)  : 78.07%
3. Sai lệch trung bình (MAE)       : 183,747 AUD
✅ Hệ thống

In [ ]:
!git config --global user.email "giangpro12332@gmail.com"
!git config --global user.name "EntiziStudio"